# FER2013 Data Exploration

Use this notebook in Colab or Kaggle to inspect the FER2013 CSV before training. The goal is to confirm the split sizes, class imbalance, and sample image quality.

## 1. Setup

Upload or mount the repository, then update `PROJECT_ROOT` and `CSV_PATH` if needed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

CSV_PATH = PROJECT_ROOT / 'data' / 'raw' / 'fer2013.csv'
CSV_PATH

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fer_project.data import EMOTION_LABELS, USAGE_TO_SPLIT

## 2. Load FER2013 CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df['Usage'].value_counts())
df['emotion_name'] = df['emotion'].map(EMOTION_LABELS)
df['split'] = df['Usage'].map(USAGE_TO_SPLIT)
df.groupby(['split', 'emotion_name']).size().unstack(fill_value=0)

## 3. Class Balance

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(data=df, x='emotion_name', hue='split', order=[EMOTION_LABELS[i] for i in range(7)])
plt.xticks(rotation=30)
plt.tight_layout()

## 4. Image Samples

In [ ]:
def pixels_to_image(pixel_string):
    return np.fromstring(pixel_string, sep=' ', dtype=np.float32).reshape(48, 48)

fig, axes = plt.subplots(7, 6, figsize=(8, 9))
for label, emotion in EMOTION_LABELS.items():
    samples = df[df['emotion'] == label].sample(6, random_state=42)
    for ax, (_, row) in zip(axes[label], samples.iterrows()):
        ax.imshow(pixels_to_image(row['pixels']), cmap='gray')
        ax.axis('off')
    axes[label][0].set_ylabel(emotion)
plt.tight_layout()

## Notes for Report

Record class imbalance, visible label ambiguity, and any low-quality images. This supports the instructor-requested error analysis.